In [1]:
import pandas as pd

# Replace with your actual file path
file_path = r"D:\Final_year_project\basic_code_files\Protein\biogrid_homoshapheains\BIOGRID-ORGANISM-5.0.254.tab3\BIOGRID-ORGANISM-Homo_sapiens-5.0.254.tab3.txt"

df = pd.read_csv(file_path, sep="\t", low_memory=False)

print("Dataset shape:", df.shape)
print(df.columns)


Dataset shape: (1349125, 37)
Index(['#BioGRID Interaction ID', 'Entrez Gene Interactor A',
       'Entrez Gene Interactor B', 'BioGRID ID Interactor A',
       'BioGRID ID Interactor B', 'Systematic Name Interactor A',
       'Systematic Name Interactor B', 'Official Symbol Interactor A',
       'Official Symbol Interactor B', 'Synonyms Interactor A',
       'Synonyms Interactor B', 'Experimental System',
       'Experimental System Type', 'Author', 'Publication Source',
       'Organism ID Interactor A', 'Organism ID Interactor B', 'Throughput',
       'Score', 'Modification', 'Qualifications', 'Tags', 'Source Database',
       'SWISS-PROT Accessions Interactor A', 'TREMBL Accessions Interactor A',
       'REFSEQ Accessions Interactor A', 'SWISS-PROT Accessions Interactor B',
       'TREMBL Accessions Interactor B', 'REFSEQ Accessions Interactor B',
       'Ontology Term IDs', 'Ontology Term Names', 'Ontology Term Categories',
       'Ontology Term Qualifier IDs', 'Ontology Term Quali

# step1 : Keep Only Physical Interactions

In [2]:
df_physical = df[df["Experimental System Type"] == "physical"].copy()

print("After keeping physical interactions:")
print(df_physical.shape)


After keeping physical interactions:
(1329740, 37)


# step2 : Extract Only Required Columns

In [3]:
ppi = df_physical[[
    "Official Symbol Interactor A",
    "Official Symbol Interactor B"
]].copy()

print("After selecting protein columns:")
print(ppi.shape)
print(ppi.head())


After selecting protein columns:
(1329740, 2)
  Official Symbol Interactor A Official Symbol Interactor B
0                       MAP2K4                         FLNC
1                         MYPN                        ACTN2
2                        ACVR1                         FNTA
3                        GATA2                          PML
4                         RPA2                        STAT3


# step3 - remove self loops

In [4]:
ppi = ppi[
    ppi["Official Symbol Interactor A"] != 
    ppi["Official Symbol Interactor B"]
]

print("After removing self-loops:", ppi.shape)



After removing self-loops: (1321113, 2)


# STEP 4 — Remove Duplicate Undirected Edges Because:(A, B) same as (B, A)



In [5]:
# Sort each pair alphabetically
ppi["sorted_pair"] = ppi.apply(
    lambda row: tuple(sorted([
        row["Official Symbol Interactor A"],
        row["Official Symbol Interactor B"]
    ])),
    axis=1
)

# Remove duplicates
ppi = ppi.drop_duplicates(subset="sorted_pair")

# Keep only original columns
ppi = ppi[[
    "Official Symbol Interactor A",
    "Official Symbol Interactor B"
]]

print("After removing duplicates:", ppi.shape)


After removing duplicates: (997442, 2)


# STEP 5 — Basic Graph Statistics

In [6]:
unique_nodes = set(ppi["Official Symbol Interactor A"]).union(
    set(ppi["Official Symbol Interactor B"])
)

num_nodes = len(unique_nodes)
num_edges = len(ppi)

print("Number of nodes:", num_nodes)
print("Number of edges:", num_edges)

density = (2 * num_edges) / (num_nodes * (num_nodes - 1))
print("Graph density:", density)


Number of nodes: 27295
Number of edges: 997442
Graph density: 0.00267773355748139


# STEP 6 — Save Cleaned Dataset

In [7]:
ppi.to_csv("cleaned_biogrid_ppi.csv", index=False)
print("Cleaned dataset saved successfully.")


Cleaned dataset saved successfully.
